# 02. Стенд пошкоджень (degradation benchmark)

**Тиждень 1.** Псуємо чисті коди контрольовано: 8 типів × 5 рівнів сили, за зразком
ImageNet-C (Hendrycks & Dietterich, 2019). Міняємо одну річ за раз, тому видно,
з якого рівня декодер «ламається».

Параметри рівнів у `src/degradations.py`. TODO: якщо якийсь тип занадто легкий
(наприклад, `jpeg`), посиль його рівні.

In [ ]:
import sys
from pathlib import Path

# ноутбук лежить у notebooks/, корінь проєкту на рівень вище
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from src.data import generate_synthetic
from src.degradations import DEGRADATIONS, LEVELS
from src.evaluate import run_benchmark, summarize

samples = generate_synthetic(n_per_format=10, seed=42)

## Галерея: один EAN-13, кожне пошкодження на рівнях 1, 3, 5

In [ ]:
image = samples[0]["image"]
names = list(DEGRADATIONS)
fig, axes = plt.subplots(len(names), 3, figsize=(10, 2 * len(names)))
for row in range(len(names)):
    for col, level in enumerate([1, 3, 5]):
        damaged = DEGRADATIONS[names[row]](image, level, seed=1)
        axes[row, col].imshow(damaged, cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(names[row] + " " + str(level), fontsize=8)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## Baseline: сирі кадри (без обробки) через усі декодери

In [ ]:
baseline = run_benchmark(samples, methods=["raw"])
baseline.to_csv(RESULTS / "02_baseline.csv", index=False)
print("Рядків:", len(baseline))

In [ ]:
summary = summarize(baseline, by=["decoder", "degradation", "level"])
summary.pivot_table(index=["decoder", "degradation"], columns="level", values="decode_rate").round(2)

## Криві: decode rate проти рівня, лінія на кожен декодер

In [ ]:
curves = summarize(baseline, by=["decoder", "degradation", "level"])
clean = curves[curves["degradation"] == "clean"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, name in zip(axes.flat, DEGRADATIONS):
    for decoder in curves["decoder"].unique():
        line = curves[(curves["degradation"] == name) & (curves["decoder"] == decoder)]
        start = clean[clean["decoder"] == decoder]
        levels = [0] + list(line["level"])
        rates = list(start["decode_rate"]) + list(line["decode_rate"])
        ax.plot(levels, rates, marker="o", label=decoder)
    ax.set_title(name)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)
axes[0, 0].legend()
plt.tight_layout()
plt.show()

**Питання для тебе:**
- Яке пошкодження найнебезпечніше і для якого декодера?
- Чому `jpeg` майже не шкодить? (підказка: розмір модуля коду і блоки 8×8 JPEG, лекція 5)
- Хибні декоди (`wrong_rate`): чи з'являються взагалі?

In [ ]:
summarize(baseline, by=["decoder"])